# Architectural "Beads" and Design Decisions

This notebook explores the concept of architectural "Beads"—discrete, composable units that encapsulate specific architectural decisions. Each bead represents a cohesive design choice that can be analyzed, documented, and composed with other beads to form complete architectural patterns.

## State Map
- [Define Beads Concept](#beads-concept)
- [Core Beads in aws-cache](#core-beads)
- [Trade-offs & Constraints](#tradeoffs)
- [Composition Patterns](#composition)

## Define Beads Concept {#beads-concept}

An architectural "Bead" is a discrete unit of design that:
1. **Encapsulates** a specific architectural decision (e.g., "single-file monolithic design")
2. **Documents** rationale, constraints, and trade-offs
3. **Composes** with other beads to form complete architectures
4. **Interfaces** clearly with adjacent beads (boundaries)

Beads prioritize **clarity, isolation, and composability** over granularity.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from enum import Enum

class BeadType(Enum):
    """Categories of architectural beads."""
    EXECUTION = "execution"  # How code runs
    STORAGE = "storage"      # How data persists
    COMMUNICATION = "communication"  # How components interact
    SAFETY = "safety"        # How we prevent errors
    PERFORMANCE = "performance"  # How we optimize

@dataclass
class Bead:
    """Represents a single architectural decision."""
    name: str
    type: BeadType
    decision: str  # What we decided
    rationale: str  # Why we decided it
    constraints: List[str] = field(default_factory=list)  # What limits this bead
    trade_offs: Dict[str, str] = field(default_factory=dict)  # what vs. what
    depends_on: List[str] = field(default_factory=list)  # Other beads this assumes

    def __repr__(self) -> str:
        return f"Bead({self.name} [{self.type.value}])"

# Example: Create a bead for aws-cache's monolithic design
monolithic_bead = Bead(
    name="Monolithic Design",
    type=BeadType.EXECUTION,
    decision="Single aws-cache file with no external dependencies",
    rationale="Portability: no pip install, no dependency conflicts, easy deployment (cp to ~/bin/)",
    constraints=[
        "All logic in one file (~500-1000 LOC)",
        "No external packages (only stdlib)",
        "No package manager required"
    ],
    trade_offs={
        "Modularity": "Monolithic (harder to test in isolation)",
        "Reuse": "Limited (can't import as library)",
        "Maintainability": "Simpler (one file to review)"
    }
)

print(monolithic_bead)
print(f"\nRationale: {monolithic_bead.rationale}")
print(f"Trade-offs: {monolithic_bead.trade_offs}")

## Core Beads in aws-cache {#core-beads}

The aws-cache architecture is built from these interrelated beads:

In [ ]:
# Define core beads for aws-cache

core_beads = [
    Bead(
        name="Monolithic Design",
        type=BeadType.EXECUTION,
        decision="Single aws-cache file with no external dependencies",
        rationale="Portability: zero pip installs, drop-in deployment",
        constraints=["~500-1000 LOC", "Stdlib only", "No package manager"],
        trade_offs={"Modularity": "Low", "Testability": "Requires integration tests"},
        depends_on=[]
    ),
    Bead(
        name="Three-Tier Classification",
        type=BeadType.SAFETY,
        decision="Classify operations (read/write) via: AWS Service Reference API → IAM heuristics → fail-safe (no cache)",
        rationale="Safety-first: never cache writes; fail-safe on unknown ops",
        constraints=["Authoritative source required", "API calls add latency"],
        trade_offs={"Performance": "API calls on first encounter", "Safety": "Guaranteed correctness"},
        depends_on=["Monolithic Design"]
    ),
    Bead(
        name="Context-Isolated Caching",
        type=BeadType.STORAGE,
        decision="Cache keyed on normalized args + profile + region; SHA256 hash for consistent keys",
        rationale="Prevent cross-account leakage; normalize arg order for consistent hits",
        constraints=["Requires profile/region extraction", "Argument sorting overhead"],
        trade_offs={"Simplicity": "Key generation logic", "Correctness": "Account isolation"},
        depends_on=["Monolithic Design"]
    ),
    Bead(
        name="JSON File Storage",
        type=BeadType.STORAGE,
        decision="Cache stored as JSON files in ~/.aws-cache/; TTL + metadata in file",
        rationale="Human-readable, portable, no DB required",
        constraints=["File I/O latency", "Manual cleanup"],
        trade_offs={"Performance": "File I/O vs. memory", "Debuggability": "Human-readable cache"},
        depends_on=["Context-Isolated Caching"]
    ),
    Bead(
        name="Subprocess Execution",
        type=BeadType.EXECUTION,
        decision="Invoke AWS CLI via subprocess.run(); pass through stdout/stderr",
        rationale="Transparent to user; no need to reimplement AWS logic",
        constraints=["Subprocess overhead", "No in-process optimization"],
        trade_offs={"Performance": "Subprocess spawn cost", "Compatibility": "Works with any AWS CLI version"},
        depends_on=["Monolithic Design"]
    )
]

# Display all beads
for i, bead in enumerate(core_beads, 1):
    print(f"{i}. {bead.name} [{bead.type.value}]")
    print(f"   Decision: {bead.decision}")
    print(f"   Depends on: {', '.join(bead.depends_on) if bead.depends_on else 'None'}")
    print()

## Trade-offs & Constraints {#tradeoffs}

Each bead makes trade-offs explicit. Understanding these is critical for future design decisions.

In [ ]:
# Analyze trade-offs across all beads

print("=" * 80)
print("ARCHITECTURAL TRADE-OFFS")
print("=" * 80)

trade_off_summary = {}

for bead in core_beads:
    print(f"\n{bead.name}:")
    print(f"  Rationale: {bead.rationale}")
    print(f"  Constraints:")
    for constraint in bead.constraints:
        print(f"    - {constraint}")
    print(f"  Trade-offs:")
    for dimension, trade_off in bead.trade_offs.items():
        print(f"    - {dimension}: {trade_off}")
        if dimension not in trade_off_summary:
            trade_off_summary[dimension] = []
        trade_off_summary[dimension].append((bead.name, trade_off))

print("\n" + "=" * 80)
print("TRADE-OFF MATRIX (by dimension)")
print("=" * 80)

for dimension, items in sorted(trade_off_summary.items()):
    print(f"\n{dimension}:")
    for bead_name, trade_off in items:
        print(f"  {bead_name}: {trade_off}")

## Bead Dependencies & Composition {#composition}

Beads depend on one another. Understanding these relationships prevents architectural conflicts.

In [ ]:
# Build a dependency graph

def build_dependency_graph(beads: List[Bead]):
    """Create adjacency list of bead dependencies."""
    graph = {bead.name: bead.depends_on for bead in beads}
    return graph

def topological_sort(beads: List[Bead]):
    """Order beads by dependency (independent first)."""
    from collections import defaultdict, deque
    
    graph = build_dependency_graph(beads)
    bead_names = {bead.name for bead in beads}
    
    # Calculate in-degrees
    in_degree = {name: 0 for name in bead_names}
    for name, deps in graph.items():
        for dep in deps:
            if dep in in_degree:
                in_degree[dep] += 1  # Parent has outgoing edge
    
    # Find nodes with no incoming edges
    queue = deque([name for name in bead_names if in_degree[name] == 0])
    sorted_order = []
    
    while queue:
        node = queue.popleft()
        sorted_order.append(node)
        
        # Find dependents
        for other_name, deps in graph.items():
            if node in deps:
                in_degree[other_name] -= 1
                if in_degree[other_name] == 0:
                    queue.append(other_name)
    
    return sorted_order

# Analyze composition
print("BEAD COMPOSITION ORDER (independent → dependent):")
print("=" * 80)
order = topological_sort(core_beads)
for i, bead_name in enumerate(order, 1):
    bead = next(b for b in core_beads if b.name == bead_name)
    print(f"{i}. {bead_name}")
    if bead.depends_on:
        print(f"   └─ depends on: {', '.join(bead.depends_on)}")
    print()

## Future Bead Candidates

Potential architectural decisions for future versions:

In [ ]:
future_beads = [
    Bead(
        name="Distributed Cache",
        type=BeadType.STORAGE,
        decision="Share cache across AWS accounts/profiles via shared backend (e.g., DynamoDB/Redis)",
        rationale="Multi-user efficiency; cache hits from colleagues",
        constraints=["External service required", "Network latency"],
        trade_offs={"Autonomy": "Requires shared service", "Efficiency": "Cross-user cache hits"},
        depends_on=["Context-Isolated Caching", "Subprocess Execution"]
    ),
    Bead(
        name="Async Prefetch",
        type=BeadType.PERFORMANCE,
        decision="Background refresh of stale cache entries before TTL expires",
        rationale="Always-fresh data; avoid user-facing latency",
        constraints=["Background daemon", "Resource usage"],
        trade_offs={"Complexity": "Daemon required", "Performance": "Fresh cache"},
        depends_on=["JSON File Storage", "Subprocess Execution"]
    ),
    Bead(
        name="Pluggable Classification",
        type=BeadType.SAFETY,
        decision="Allow custom classification plugins for org-specific operations",
        rationale="Support custom AWS services or internal CLIs",
        constraints=["Plugin interface, validation"],
        trade_offs={"Complexity": "Plugin system", "Flexibility": "Custom classifications"},
        depends_on=["Three-Tier Classification"]
    )
]

print("FUTURE BEAD CANDIDATES:")
print("=" * 80)
for bead in future_beads:
    print(f"\n{bead.name} [{bead.type.value}]")
    print(f"  Rationale: {bead.rationale}")
    print(f"  Depends on: {', '.join(bead.depends_on)}")

## Summary

The aws-cache architecture is built from **5 core beads**:
1. **Monolithic Design** (Execution): Single file, zero dependencies
2. **Three-Tier Classification** (Safety): Safety-first operation classification
3. **Context-Isolated Caching** (Storage): Profile/region-scoped cache keys
4. **JSON File Storage** (Storage): Human-readable persistent cache
5. **Subprocess Execution** (Execution): Transparent AWS CLI invocation

Each bead makes **explicit trade-offs** and **clear assumptions** about other beads. Future changes should be evaluated against this beads model to ensure **coherence** and **safety**.